# Sample 01: 基本的な宣言的構造体 (`@binary_struct`)

`binary_master` の最も基本的な機能である宣言的バイナリ構造体定義のチュートリアルです。

### 学べる内容
- `@binary_struct` クラスデコレータによる構造体モデリング
- 基本プリミティブ型 (`UInt8`, `UInt16`, `UInt32`, `Float32`, `Bool`) と固定長文字列 (`FixedString`)
- 静的バイトサイズの確認 (`sizeof()`, `.binary_size`)
- フィールドバイトオフセットの取得 (`offsetof()`)
- シリアライズ (`to_bytes()`) とデシリアライズ (`read_struct()`, `from_bytes()`)
- エンディアンの制御 (`Endian.LITTLE` / `Endian.BIG`)

In [1]:
from binary_master import (
    Bool,
    Endian,
    FixedString,
    Float32,
    UInt8,
    UInt16,
    UInt32,
    binary_struct,
    hexdump,
    read_struct,
    sizeof,
)

## 1. 構造体クラスの定義

Python の型アノテーションを用いて各フィールドの型を指定します。
クラス docstring やインラインコメントは、仕様書生成時にも自動反映されます。

In [2]:
@binary_struct(endian="little")
class PlayerProfile:
    """プレイヤープロファイルヘッダー."""
    magic: UInt32          # シグネチャ: 0x504C4159 ('PLAY')
    player_id: UInt16      # プレイヤーID
    level: UInt8           # レベル
    lives: UInt8           # 残機
    score: UInt32          # スコア
    health_ratio: Float32  # 体力比率 (0.0 - 1.0)
    is_vip: Bool           # VIPフラグ (1バイト真偽値: 0x01=True, 0x00=False)
    tag: FixedString[4]    # 4文字のクランタグ (例: "PROG")

# 静的サイズの取得
print(f"PlayerProfile クラスの静的サイズ: {sizeof(PlayerProfile)} バイト (sizeof)")
print(f"PlayerProfile クラスの静的サイズ: {PlayerProfile.binary_size} バイト (.binary_size)")

PlayerProfile クラスの静的サイズ: 21 バイト (sizeof)
PlayerProfile クラスの静的サイズ: 21 バイト (.binary_size)

## 2. フィールドオフセットの確認 (`offsetof`)

C言語の `offsetof(struct, member)` と同様に、各フィールドの開始バイトオフセットを静的に取得できます。

In [3]:
print("各フィールドの開始バイトオフセット:")
for field in ["magic", "player_id", "level", "lives", "score", "health_ratio", "is_vip", "tag"]:
    print(f"  {field:14s}: offset {PlayerProfile.offsetof(field):2d} バイト (0x{PlayerProfile.offsetof(field):02X})")

各フィールドの開始バイトオフセット:
  magic         : offset  0 バイト (0x00)
  player_id     : offset  4 バイト (0x04)
  level         : offset  6 バイト (0x06)
  lives         : offset  7 バイト (0x07)
  score         : offset  8 バイト (0x08)
  health_ratio  : offset 12 バイト (0x0C)
  is_vip        : offset 16 バイト (0x10)
  tag           : offset 17 バイト (0x11)

## 3. インスタンス化とシリアライズ (`to_bytes`)

インスタンスを作成し、`.to_bytes()` を呼び出すことで生のバイナリデータ（`bytes`）を生成します。
`hexdump` を使って、生成されたバイト列の配置を視覚的に確認できます。

In [4]:
player = PlayerProfile(
    magic=0x59414C50,  # 'PLAY' in little endian
    player_id=1042,
    level=50,
    lives=3,
    score=999999,
    health_ratio=0.85,
    is_vip=True,
    tag="PROG",
)

raw_bytes = player.to_bytes()
print(f"シリアライズ結果 ({len(raw_bytes)} バイト):")
print(hexdump(raw_bytes, annotate=True))

シリアライズ結果 (21 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  50 4c 41 59 12 04 32 03  3f 42 0f 00 9a 99 59 3f  |PLAY..2.?B....Y?|
00000010  01 50 52 4f 47                                    |.PROG           |
  [Total: 21 bytes (`0x0015`)]

## 4. バイナリからのデシリアライズ (`read_struct` / `from_bytes`)

`read_struct(PlayerProfile, raw_bytes)` または `PlayerProfile.from_bytes(raw_bytes)` で構造体を復元します。

In [5]:
restored = read_struct(PlayerProfile, raw_bytes)
print(f"player_id:    {restored.player_id}")
print(f"level:        {restored.level}")
print(f"lives:        {restored.lives}")
print(f"score:        {restored.score}")
print(f"health_ratio: {restored.health_ratio:.2f}")
print(f"is_vip:       {restored.is_vip} (Python bool)")
print(f"tag:          {restored.tag!r}")

assert restored.is_vip is True
assert restored.tag == "PROG"
assert restored.score == 999999
print("\nデシリアライズ検証成功！")

player_id:    1042
level:        50
lives:        3
score:        999999
health_ratio: 0.85
is_vip:       True (Python bool)
tag:          'PROG'

デシリアライズ検証成功！

## 5. エンディアンの動的オーバーライド (`Endian.BIG`)

シリアライズおよびデシリアライズ時に、構造体定義のデフォルトに関わらずエンディアンを動的に上書きできます。

In [6]:
# ビッグエンディアンで書き込み
data_be = player.to_bytes(endian=Endian.BIG)
print(f"Big-endian シリアライズ ({len(data_be)} バイト): {data_be.hex(' ')}")

# ビッグエンディアンで読み込み復元
restored_be = read_struct(PlayerProfile, data_be, endian=Endian.BIG)
assert restored_be.player_id == player.player_id
assert restored_be.score == player.score
assert restored_be.is_vip is True
print("Big-endian ラウンドトリップ成功！")

Big-endian シリアライズ (21 バイト): 59 41 4c 50 04 12 32 03 00 0f 42 3f 3f 59 99 9a 01 50 52 4f 47
Big-endian ラウンドトリップ成功！

## 6. 静的型チェッカー対応 (`BinaryStruct`) と辞書/JSON変換 (`to_dict`)

IDE のコード補完や静的型チェッカー（`ty`, `mypy`, `pyright`）で動的属性の警告を防ぐには、基底クラス `BinaryStruct` を継承します。
また、`.to_dict()` や `.to_json()` で構造体を辞書や JSON 文字列に簡単に相互変換できます。

In [7]:
from binary_master import BinaryStruct


@binary_struct
class ServerConfig(BinaryStruct):
    port: UInt16 = 8080
    max_clients: UInt16 = 100

config = ServerConfig()
# IDE の補完が効き、静的型警告もゼロ
config_dict = config.to_dict()
print(f"辞書表現: {config_dict}")
restored_config = ServerConfig.from_dict(config_dict)
assert restored_config.port == 8080
print("BinaryStruct と辞書変換の検証成功！")

辞書表現: {'port': 8080, 'max_clients': 100}
BinaryStruct と辞書変換の検証成功！